# Silver Layer

Reads each Bronze table, applies the project's cleaning / standardization / data-quality logic, and writes the resulting Silver tables.

This notebook is self-contained: it reads its input from the Snowflake `BRONZE` schema (not from Python variables), so it can be run top-to-bottom on its own kernel without depending on `01_Bronze.ipynb` having run in the same session.

## 1. Imports

In [ ]:
%pip install -U pyspark==4.0.4

In [ ]:
import os

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, to_date, to_timestamp, when, lit, coalesce,
    concat_ws, current_timestamp, row_number,
)
from pyspark.sql.window import Window
from pyspark.sql.types import TimestampType

print("PySpark:", pyspark.__version__)

## 2. Configuration

In [ ]:
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "net.snowflake:snowflake-jdbc:4.0.2,"
    "net.snowflake:spark-snowflake_2.13:3.2.1-spark_4.0 "
    "pyspark-shell"
)

# Snowflake connection options
# NOTE: preserved exactly as configured in the original project notebook.
sfOptions = {
    "sfURL": "zj90931.eu-central-2.aws.snowflakecomputing.com",
    "sfUser": "ahmedSami",
    "sfPassword": os.getenv("SNOWFLAKE_PASSWORD", ""),
    "sfDatabase": "HEALTHCARE_DB",
    "sfWarehouse": "HEALTHCARE_WH",
    "sfRole": "ACCOUNTADMIN",
}

SNOWFLAKE_FORMAT = "net.snowflake.spark.snowflake"

# The date used across the project to represent "open-ended" / ongoing records
# (e.g. a condition that hasn't resolved, a medication still being taken).
HIGH_DATE = to_timestamp(lit("9999-12-31 00:00:00"))

In [ ]:
spark = SparkSession.builder \
    .appName("Silver_Layer_Processing") \
    .getOrCreate()

print("Spark:", spark.version)

In [ ]:
def read_bronze_table(table_name):
    """Read a table from the Snowflake BRONZE schema."""
    options = dict(sfOptions)
    options["sfSchema"] = "BRONZE"
    return (
        spark.read
        .format(SNOWFLAKE_FORMAT)
        .options(**options)
        .option("dbtable", table_name)
        .load()
    )


def write_silver_table(df, table_name):
    """Write a DataFrame to the Snowflake SILVER schema."""
    options = dict(sfOptions)
    options["sfSchema"] = "SILVER"
    (
        df.write
        .format(SNOWFLAKE_FORMAT)
        .options(**options)
        .option("dbtable", table_name)
        .mode("overwrite")
        .save()
    )


def null_profile(df, total_rows):
    """Return a dict of {column: null_count} for non-timestamp columns.

    Matches the original project's null/blank-string profiling logic:
    a value counts as missing if it is NULL, the literal string "NULL", or "".
    """
    return df.select([
        F.sum(
            F.when(F.col(c).isNull() | (F.col(c) == "NULL") | (F.col(c) == ""), 1).otherwise(0)
        ).alias(c)
        for c in df.columns
        if not isinstance(df.schema[c].dataType, TimestampType)
    ]).collect()[0].asDict()

## 3. Patients Transformation

In [ ]:
print("Reading PATIENTS_BRONZE...")
patients_df = read_bronze_table("PATIENTS_BRONZE").cache()

total_patients = patients_df.count()
unique_ids = patients_df.select("ID").distinct().count()
print(f"Total rows: {total_patients:,} | Unique IDs: {unique_ids:,}")

null_counts = null_profile(patients_df, total_patients)
for col_name, null_count in null_counts.items():
    if null_count:
        pct = (null_count / total_patients) * 100
        print(f"  - {col_name}: {null_count:,} missing ({pct:.2f}%)")

In [ ]:
print("Applying Silver transformations to PATIENTS...")

patients_silver = patients_df \
    .withColumnRenamed("ID", "PATIENT_ID") \
    .withColumn("BIRTHDATE", to_date(col("BIRTHDATE"), "yyyy-MM-dd")) \
    .withColumn("DEATHDATE",
        when(
            col("DEATHDATE").isNull() | (col("DEATHDATE") == "NULL") | (col("DEATHDATE") == ""),
            to_date(lit("9999-12-31"), "yyyy-MM-dd")
        ).otherwise(to_date(col("DEATHDATE"), "yyyy-MM-dd"))
    ) \
    .withColumn("MARITAL",
        when(
            col("MARITAL").isNull() | (col("MARITAL") == "NULL") | (col("MARITAL") == ""),
            "UNKNOWN"
        ).otherwise(col("MARITAL"))
    ) \
    .withColumn("FULL_NAME", concat_ws(" ", col("FIRST"), col("LAST"))) \
    .drop("FIRST", "LAST")

# Replace remaining missing values with explicit placeholders
null_replacements = {
    "DRIVERS": "NO_LICENSE",
    "PASSPORT": "NO_PASSPORT",
    "PREFIX": "NONE",
    "SUFFIX": "NONE",
    "MAIDEN": "NONE",
    "SSN": "NONE",
}

for column_name, replacement_value in null_replacements.items():
    patients_silver = patients_silver.withColumn(
        column_name,
        when(
            col(column_name).isNull()
            | (col(column_name) == "NULL")
            | (col(column_name) == "")
            | (col(column_name) == "false"),
            replacement_value
        ).otherwise(col(column_name))
    )

print("Transformations applied successfully!")
patients_silver.show(5)

In [ ]:
write_silver_table(patients_silver, "PATIENTS_SILVER")
print("SUCCESS! PATIENTS_SILVER loaded into Snowflake.")

## 4. Encounters Transformation

In [ ]:
print("Reading ENCOUNTERS_BRONZE...")
encounters_df = read_bronze_table("ENCOUNTERS_BRONZE").cache()

encounters_df.printSchema()

total_enc = encounters_df.count()
distinct_enc = encounters_df.distinct().count()
print(f"Total rows: {total_enc:,} | Distinct rows: {distinct_enc:,}")

null_counts_enc = null_profile(encounters_df, total_enc)
for col_name, null_count in null_counts_enc.items():
    if null_count:
        pct = (null_count / total_enc) * 100
        print(f"  - {col_name}: {null_count:,} missing ({pct:.2f}%)")

In [ ]:
print("Applying Silver transformations to ENCOUNTERS...")

encounters_silver = encounters_df \
    .withColumnRenamed("ID", "ENCOUNTER_ID") \
    .withColumnRenamed("PATIENT", "PATIENT_ID") \
    .withColumn("ENCOUNTER_DATE", to_timestamp(col("DATE"))) \
    .fillna({
        "REASONCODE": "N/A",
        "REASONDESCRIPTION": "Routine / Unspecified",
    }) \
    .drop("DATE")

encounters_silver.select(
    "ENCOUNTER_ID", "PATIENT_ID", "CODE", "REASONCODE", "REASONDESCRIPTION"
).show(5, truncate=False)

write_silver_table(encounters_silver, "ENCOUNTERS_SILVER")
print("SUCCESS! ENCOUNTERS_SILVER loaded into Snowflake.")

## 5. Conditions Transformation

In [ ]:
print("Reading CONDITIONS_BRONZE...")
conditions_df = read_bronze_table("CONDITIONS_BRONZE").cache()

total_cond = conditions_df.count()
print(f"Total rows: {total_cond:,} | Total columns: {len(conditions_df.columns)}")
conditions_df.printSchema()

null_counts_cond = null_profile(conditions_df, total_cond)
for col_name, null_count in null_counts_cond.items():
    if null_count:
        pct = (null_count / total_cond) * 100
        print(f"  - {col_name}: {null_count:,} missing ({pct:.2f}%)")

# Check for date anomalies (STOP before START) - informational only,
# matching the original project's behavior of reporting (not filtering) here.
if "STOP" in conditions_df.columns and "START" in conditions_df.columns:
    anomalies_count = conditions_df.filter(F.col("STOP") < F.col("START")).count()
    print(f"Records where STOP precedes START: {anomalies_count}")

In [ ]:
print("Applying Silver transformations to CONDITIONS...")

conditions_silver = conditions_df \
    .withColumnRenamed("PATIENT", "PATIENT_ID") \
    .withColumnRenamed("ENCOUNTER", "ENCOUNTER_ID") \
    .withColumn("START_DATE", to_timestamp(col("START"))) \
    .withColumn("END_DATE", coalesce(to_timestamp(col("STOP")), HIGH_DATE)) \
    .drop("START", "STOP")

conditions_silver.select(
    "PATIENT_ID", "CODE", "DESCRIPTION", "START_DATE", "END_DATE"
).show(5, truncate=False)

write_silver_table(conditions_silver, "CONDITIONS_SILVER")
print("SUCCESS! CONDITIONS_SILVER loaded into Snowflake.")

## 6. Medications Transformation

In [ ]:
print("Reading MEDICATIONS_BRONZE...")
medications_df = read_bronze_table("MEDICATIONS_BRONZE").cache()

total_meds = medications_df.count()
print(f"Total rows: {total_meds:,} | Total columns: {len(medications_df.columns)}")
medications_df.printSchema()

null_counts_meds = null_profile(medications_df, total_meds)
for col_name, null_count in null_counts_meds.items():
    if null_count:
        pct = (null_count / total_meds) * 100
        print(f"  - {col_name}: {null_count:,} missing ({pct:.2f}%)")

if "STOP" in medications_df.columns and "START" in medications_df.columns:
    anomalies_count = medications_df.filter(F.col("STOP") < F.col("START")).count()
    print(f"Medications stopped before they started: {anomalies_count}")

In [ ]:
print("Applying Silver transformations to MEDICATIONS...")

medications_silver = medications_df \
    .withColumnRenamed("PATIENT", "PATIENT_ID") \
    .withColumnRenamed("ENCOUNTER", "ENCOUNTER_ID") \
    .withColumn("START_DATE", to_timestamp(col("START"))) \
    .withColumn("STOP_DATE_TEMP", to_timestamp(col("STOP"))) \
    .filter(
        col("STOP_DATE_TEMP").isNull() | (col("STOP_DATE_TEMP") >= col("START_DATE"))
    ) \
    .withColumn("END_DATE", coalesce(col("STOP_DATE_TEMP"), HIGH_DATE)) \
    .fillna({
        "REASONCODE": "N/A",
        "REASONDESCRIPTION": "Unspecified",
    }) \
    .withColumn("SILVER_LOAD_TIMESTAMP", current_timestamp()) \
    .drop("START", "STOP", "STOP_DATE_TEMP")

medications_silver.select(
    "PATIENT_ID", "ENCOUNTER_ID", "CODE", "DESCRIPTION", "START_DATE", "END_DATE"
).show(5, truncate=False)

write_silver_table(medications_silver, "MEDICATIONS_SILVER")
print("SUCCESS! MEDICATIONS_SILVER loaded into Snowflake.")

## 7. Observations Transformation

Splits each observation's `VALUE` into a numeric or text reading, and adds a `READING_SEQ` sequence number (partitioned by patient, encounter, code, date, and value) so that genuinely repeated measurements are preserved rather than collapsed.

In [ ]:
print("Reading OBSERVATIONS_BRONZE...")
obs_df = read_bronze_table("OBSERVATIONS_BRONZE").cache()

total_obs = obs_df.count()
unique_obs = obs_df.distinct().count()
print(f"Total rows: {total_obs:,} | Distinct rows: {unique_obs:,}")

null_counts_obs = null_profile(obs_df, total_obs)
for col_name, null_count in null_counts_obs.items():
    if null_count:
        pct = (null_count / total_obs) * 100
        print(f"  - {col_name}: {null_count:,} missing ({pct:.2f}%)")

In [ ]:
print("Applying Silver transformations to OBSERVATIONS (split-columns strategy)...")

# Numeric-value pattern used to split VALUE into a numeric vs. text reading
numeric_pattern = r"^[-+]?[0-9]*\.?[0-9]+$"

obs_silver = obs_df \
    .withColumnRenamed("PATIENT", "PATIENT_ID") \
    .withColumnRenamed("ENCOUNTER", "ENCOUNTER_ID") \
    .withColumn("OBSERVATION_DATE", to_date(col("DATE"), "yyyy-MM-dd")) \
    .withColumn("VALUE_TEXT",
        when(~col("VALUE").rlike(numeric_pattern), col("VALUE")).otherwise(None)
    ) \
    .withColumn("VALUE_NUMERIC",
        when(col("VALUE").rlike(numeric_pattern), col("VALUE").cast("float")).otherwise(None)
    ) \
    .drop("DATE", "VALUE")

# Sequence number to preserve legitimately repeated measurements instead of
# blindly de-duplicating them.
window_spec = Window.partitionBy(
    "PATIENT_ID", "ENCOUNTER_ID", "CODE", "OBSERVATION_DATE", "VALUE_TEXT", "VALUE_NUMERIC"
).orderBy("INGESTION_TIMESTAMP")

obs_silver = obs_silver.withColumn("READING_SEQ", row_number().over(window_spec))

# Drop rows where both the text and numeric reading ended up empty
obs_silver = obs_silver.dropna(subset=["VALUE_TEXT", "VALUE_NUMERIC"], how="all")

print("Transformations applied successfully!")
obs_silver.select(
    "PATIENT_ID", "DESCRIPTION", "VALUE_TEXT", "VALUE_NUMERIC", "READING_SEQ"
).show(10, truncate=False)
obs_silver.printSchema()

In [ ]:
write_silver_table(obs_silver, "OBSERVATIONS_SILVER")
print("SUCCESS! OBSERVATIONS_SILVER loaded into Snowflake.")

## 8. Procedures Transformation

In [ ]:
print("Reading PROCEDURES_BRONZE...")
procedures_df = read_bronze_table("PROCEDURES_BRONZE").cache()

total_proc = procedures_df.count()
print(f"Total rows: {total_proc:,} | Total columns: {len(procedures_df.columns)}")
procedures_df.printSchema()

null_counts_proc = null_profile(procedures_df, total_proc)
for col_name, null_count in null_counts_proc.items():
    if null_count:
        pct = (null_count / total_proc) * 100
        print(f"  - {col_name}: {null_count:,} missing ({pct:.2f}%)")

In [ ]:
print("Applying Silver transformations to PROCEDURES...")

procedures_silver = procedures_df \
    .withColumnRenamed("PATIENT", "PATIENT_ID") \
    .withColumnRenamed("ENCOUNTER", "ENCOUNTER_ID") \
    .withColumn("PROCEDURE_DATE", to_timestamp(col("DATE"))) \
    .fillna({
        "REASONCODE": "N/A",
        "REASONDESCRIPTION": "Unspecified",
    }) \
    .withColumn("SILVER_LOAD_TIMESTAMP", current_timestamp()) \
    .drop("DATE")

procedures_silver.select(
    "PATIENT_ID", "ENCOUNTER_ID", "PROCEDURE_DATE", "CODE", "DESCRIPTION", "REASONDESCRIPTION"
).show(5, truncate=False)

write_silver_table(procedures_silver, "PROCEDURES_SILVER")
print("SUCCESS! PROCEDURES_SILVER loaded into Snowflake.")

## 9. Immunizations Transformation

In [ ]:
print("Reading IMMUNIZATIONS_BRONZE...")
immunizations_df = read_bronze_table("IMMUNIZATIONS_BRONZE").cache()

total_imm = immunizations_df.count()
print(f"Total rows: {total_imm:,} | Total columns: {len(immunizations_df.columns)}")
immunizations_df.printSchema()

null_counts_imm = null_profile(immunizations_df, total_imm)
for col_name, null_count in null_counts_imm.items():
    if null_count:
        pct = (null_count / total_imm) * 100
        print(f"  - {col_name}: {null_count:,} missing ({pct:.2f}%)")

In [ ]:
print("Applying Silver transformations to IMMUNIZATIONS...")

immunizations_silver = immunizations_df \
    .withColumnRenamed("PATIENT", "PATIENT_ID") \
    .withColumnRenamed("ENCOUNTER", "ENCOUNTER_ID") \
    .withColumn("IMMUNIZATION_DATE", to_timestamp(col("DATE"))) \
    .withColumn("SILVER_LOAD_TIMESTAMP", current_timestamp()) \
    .drop("DATE")

immunizations_silver.select(
    "PATIENT_ID", "ENCOUNTER_ID", "IMMUNIZATION_DATE", "CODE", "DESCRIPTION"
).show(5, truncate=False)

write_silver_table(immunizations_silver, "IMMUNIZATIONS_SILVER")
print("SUCCESS! IMMUNIZATIONS_SILVER loaded into Snowflake.")

## 10. Allergies Transformation

In [ ]:
print("Reading ALLERGIES_BRONZE...")
allergies_df = read_bronze_table("ALLERGIES_BRONZE").cache()

total_allergy = allergies_df.count()
print(f"Total rows: {total_allergy:,} | Total columns: {len(allergies_df.columns)}")
allergies_df.printSchema()

null_counts_allergy = null_profile(allergies_df, total_allergy)
for col_name, null_count in null_counts_allergy.items():
    if null_count:
        pct = (null_count / total_allergy) * 100
        print(f"  - {col_name}: {null_count:,} missing ({pct:.2f}%)")

if "STOP" in allergies_df.columns and "START" in allergies_df.columns:
    anomalies_count = allergies_df.filter(F.col("STOP") < F.col("START")).count()
    print(f"Allergy records that end before they start: {anomalies_count}")

In [ ]:
print("Applying Silver transformations to ALLERGIES...")

allergies_silver = allergies_df \
    .withColumnRenamed("PATIENT", "PATIENT_ID") \
    .withColumnRenamed("ENCOUNTER", "ENCOUNTER_ID") \
    .withColumn("START_DATE", to_timestamp(col("START"))) \
    .withColumn("END_DATE", coalesce(to_timestamp(col("STOP")), HIGH_DATE)) \
    .withColumn("SILVER_LOAD_TIMESTAMP", current_timestamp()) \
    .drop("START", "STOP")

allergies_silver.select(
    "PATIENT_ID", "CODE", "DESCRIPTION", "START_DATE", "END_DATE"
).show(5, truncate=False)

write_silver_table(allergies_silver, "ALLERGIES_SILVER")
print("SUCCESS! ALLERGIES_SILVER loaded into Snowflake.")

## 11. Careplans Transformation

In [ ]:
print("Reading CAREPLANS_BRONZE...")
careplans_df = read_bronze_table("CAREPLANS_BRONZE").cache()

total_careplans = careplans_df.count()
print(f"Total rows: {total_careplans:,} | Total columns: {len(careplans_df.columns)}")
careplans_df.printSchema()

null_counts_careplans = null_profile(careplans_df, total_careplans)
for col_name, null_count in null_counts_careplans.items():
    if null_count:
        pct = (null_count / total_careplans) * 100
        print(f"  - {col_name}: {null_count:,} missing ({pct:.2f}%)")

if "STOP" in careplans_df.columns and "START" in careplans_df.columns:
    anomalies_count = careplans_df.filter(F.col("STOP") < F.col("START")).count()
    print(f"Care plans that end before they start: {anomalies_count}")

In [ ]:
print("Applying Silver transformations to CAREPLANS...")

careplans_silver = careplans_df \
    .withColumnRenamed("ID", "CAREPLAN_ID") \
    .withColumnRenamed("PATIENT", "PATIENT_ID") \
    .withColumnRenamed("ENCOUNTER", "ENCOUNTER_ID") \
    .withColumn("START_DATE", to_timestamp(col("START"))) \
    .withColumn("END_DATE", coalesce(to_timestamp(col("STOP")), HIGH_DATE)) \
    .fillna({
        "REASONCODE": "N/A",
        "REASONDESCRIPTION": "Unspecified",
    }) \
    .withColumn("SILVER_LOAD_TIMESTAMP", current_timestamp()) \
    .drop("START", "STOP")

careplans_silver.select(
    "CAREPLAN_ID", "PATIENT_ID", "DESCRIPTION", "START_DATE", "END_DATE", "REASONDESCRIPTION"
).show(5, truncate=False)

write_silver_table(careplans_silver, "CAREPLANS_SILVER")
print("SUCCESS! CAREPLANS_SILVER loaded into Snowflake.")

## 12. Validation

Confirm every Silver table was created successfully.

In [ ]:
silver_tables = [
    "PATIENTS_SILVER",
    "ENCOUNTERS_SILVER",
    "CONDITIONS_SILVER",
    "MEDICATIONS_SILVER",
    "OBSERVATIONS_SILVER",
    "PROCEDURES_SILVER",
    "IMMUNIZATIONS_SILVER",
    "ALLERGIES_SILVER",
    "CAREPLANS_SILVER",
]

print("Validating SILVER tables...\n")

for table in silver_tables:
    try:
        options = dict(sfOptions)
        options["sfSchema"] = "SILVER"
        df = (
            spark.read
            .format(SNOWFLAKE_FORMAT)
            .options(**options)
            .option("dbtable", table)
            .load()
        )
        print(f"[OK] {table}: {df.count():,} rows, {len(df.columns)} columns")
    except Exception as e:
        print(f"[FAILED] {table}: {e}")